# Component I. Vector Indexing and Embedding

### 1. Install the required libraries

I begin by preparing the Colab environment with the packages used throughout the indexing workflow. `faiss-cpu` provides the vector-search functions, `sentence-transformers` supplies the embedding model, and `pypdf` reads text from the uploaded handbook. The Transformers and SentencePiece packages support the language model used later, while pandas and openpyxl handle the chunk-inspection workbook.

Keeping installation separate from the rest of the code makes setup problems easier to identify. If a package cannot be installed, I can resolve that issue before uploading the PDF or creating any vectors. The quiet `-q` option also keeps the output shorter, although Colab may still display important warnings or dependency messages.


In [1]:

# ============================================================
# RAG DOCUMENT PROCESSING PIPELINE
# PDF EXTRACTION, CHUNKING, EMBEDDINGS, FAISS, AND RETRIEVAL
# WITH DOWNLOADABLE EXCEL CHUNK INSPECTION
# Google Colab implementation
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    "transformers>=4.51.0" \
    sentencepiece \
    pandas \
    openpyxl




### 2. Import the indexing tools

After installing the dependencies, this cell imports the particular classes and modules used by Component I. `files` connects the notebook to Colab's upload and download interface, `PdfReader` extracts text from the PDF, and `SentenceTransformer` creates semantic embeddings. FAISS, NumPy, pandas, pickle, regular expressions, and file-path utilities support the remaining indexing and saving steps.

The three openpyxl style classes are included because the notebook does more than create a basic spreadsheet. They allow the exported chunk workbook to use readable headings, alignment, and formatting. Seeing all imports together also gives a useful overview of what the indexing component depends on before its functions are defined.


In [2]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment




### 3. Set the indexing parameters

This cell places the main experimental settings where they can be reviewed and changed without searching through the rest of the notebook. `CHUNK_SIZE` determines the maximum number of characters placed in a chunk, and `CHUNK_OVERLAP` repeats part of the previous chunk so that information near a boundary is less likely to lose its context. `TOP_K` gives the default number of results returned by the basic retrieval function.

The embedding model is set through `EMBEDDING_MODEL_NAME`. Using the same named model later for the chunks and the questions is essential because both need to be represented in the same vector space. I kept these settings visible because even a small adjustment to chunking can change which provisions are retrieved for a question.


In [3]:
# ============================================================
# 3. INDEXING PARAMETERS
# ============================================================

# Change these values to experiment with chunking.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"




### Define the Chunking Configurations

This section lists the twelve chunk-size and overlap combinations used in the experiment. Four chunk sizes are tested using zero, ten percent, and twenty percent overlap. Keeping these combinations organized in one list makes it easier to run the same indexing and retrieval process for every configuration without manually changing the values each time. The final configuration, C12, represents the original baseline of 1,000 characters with an overlap of 200 characters.

In [4]:
# Twelve chunking configurations for the Part 2 experiment
CHUNK_CONFIGURATIONS = [
    {"config": "C1",  "chunk_size": 300,  "chunk_overlap": 0},
    {"config": "C2",  "chunk_size": 300,  "chunk_overlap": 30},
    {"config": "C3",  "chunk_size": 300,  "chunk_overlap": 60},

    {"config": "C4",  "chunk_size": 500,  "chunk_overlap": 0},
    {"config": "C5",  "chunk_size": 500,  "chunk_overlap": 50},
    {"config": "C6",  "chunk_size": 500,  "chunk_overlap": 100},

    {"config": "C7",  "chunk_size": 700,  "chunk_overlap": 0},
    {"config": "C8",  "chunk_size": 700,  "chunk_overlap": 70},
    {"config": "C9",  "chunk_size": 700,  "chunk_overlap": 140},

    {"config": "C10", "chunk_size": 1000, "chunk_overlap": 0},
    {"config": "C11", "chunk_size": 1000, "chunk_overlap": 100},
    {"config": "C12", "chunk_size": 1000, "chunk_overlap": 200}
]

### 4. Validate and display the configuration

Before processing the document, the code checks whether the selected values can produce a valid chunking process. A chunk size of zero or less would not create usable passages, and a negative overlap would not make sense. The overlap must also remain smaller than the chunk size because the later step size is calculated by subtracting one from the other. If both were equal, the chunking loop would never move forward.

The same type of check is applied to `TOP_K`, since asking the index to return zero results would make the retrieval stage ineffective. After validation, the settings are printed in a clear summary. I find this useful when comparing experiments because the notebook output itself records which configuration was active during that run.


In [5]:
# ============================================================
# 4. VALIDATE CHUNK PARAMETERS
# ============================================================

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)




RAG INDEXING CONFIGURATION
Chunk size       : 1000
Chunk overlap    : 200
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


### 5. Upload and validate the PDF

This section starts the interactive document input through `files.upload()`. Colab returns the uploaded files as a dictionary, so the code verifies that at least one file was received and then uses the first filename as `pdf_file`. It also checks the extension before continuing, since every later extraction step expects a PDF.

These early checks make the workflow more understandable when something goes wrong. Instead of producing a less helpful error deep in `PdfReader`, the notebook immediately explains whether no document was selected or whether the chosen file was not a PDF. The selected filename is printed so I can confirm that the intended handbook is about to be indexed.


In [6]:
# ============================================================
# 5. UPLOAD PDF DOCUMENT
# ============================================================

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)






UPLOAD DOCUMENT


Saving Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition.pdf to Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition.pdf

Selected file:
Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition.pdf


### 6. Extract and normalize page text

`PdfReader` opens the uploaded document, and the loop processes its pages in their original order. For every page, `extract_text()` attempts to recover the readable text layer. Pages that contain usable text are stored together with a one-based PDF page number, which is the same numbering later shown in retrieval results and citations.

The regular expression `re.sub(r"\s+", " ", page_text)` replaces repeated spaces, line breaks, and other whitespace with a single space. PDF text often contains formatting breaks that are visually meaningful on the page but distracting during semantic search. Normalizing them gives the chunker cleaner sentences while preserving the words themselves.

I consider the page metadata just as important as the extracted text. Without it, FAISS could still retrieve similar passages, but the final answer could not be traced back to a specific part of the handbook.


In [7]:
# ============================================================
# 6. EXTRACT TEXT FROM PDF
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))






EXTRACTING PDF TEXT
PDF pages: 81
Pages containing text: 73


### 7. Confirm that extraction succeeded

This guard checks the `pages` collection before chunking begins. If the list is empty, the notebook stops and explains that the PDF may contain scanned images rather than selectable text. In that situation, ordinary PDF extraction is not enough and an optical character recognition step would be needed.

Stopping at this point prevents a chain of misleading results. An empty document would otherwise lead to no chunks, no embeddings, and an invalid FAISS index. The direct message makes the actual problem easier to recognize.


In [8]:
# ============================================================
# 7. CHECK WHETHER TEXT WAS EXTRACTED
# ============================================================

if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )




### 8. Create overlapping document chunks

The `create_chunks()` function turns the extracted pages into smaller passages that can be embedded and retrieved. It accepts the page collection together with the chosen size and overlap, then calculates `step_size` as `chunk_size - chunk_overlap`. With a size of 1,000 and an overlap of 200, the starting position moves forward by 800 characters for each new chunk.

Chunking is performed independently for every PDF page. The `start` and `end` positions slice the normalized page text, and `.strip()` removes unnecessary spaces around the result. A chunk is added only when it contains text. This page-by-page approach is useful because a retrieved chunk always has one unambiguous page number and never joins the end of one page to the beginning of another.

Each saved dictionary contains a sequential `chunk_id`, the original `page` number, and the actual `text`. The identifier gives every passage a stable label, while the page supports source checking. After one chunk is stored, `start` advances by the calculated step size, preserving the planned overlap until the end of the page is reached.

This function is central to the activity because chunk boundaries influence both retrieval precision and the amount of context available to the language model. A chunk that is too short may separate a rule from its condition, while a very long chunk may include several similar rules that are harder to distinguish.


In [9]:
# ============================================================
# 8. CREATE DOCUMENT CHUNKS
# ============================================================

def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


# Store the chunks generated for every configuration.
chunks_by_configuration = {}

for configuration in CHUNK_CONFIGURATIONS:

    config_id = configuration["config"]
    chunk_size = configuration["chunk_size"]
    chunk_overlap = configuration["chunk_overlap"]

    configuration_chunks = create_chunks(
        pages,
        chunk_size,
        chunk_overlap
    )

    chunks_by_configuration[config_id] = configuration_chunks

    print(
        f"{config_id}: "
        f"chunk size = {chunk_size}, "
        f"overlap = {chunk_overlap}, "
        f"total chunks = {len(configuration_chunks)}"
    )


# Keep C12 as the active baseline for the notebook's original cells.
chunks = chunks_by_configuration["C12"]

print("\nActive baseline configuration: C12")
print("Active baseline chunks:", len(chunks))




C1: chunk size = 300, overlap = 0, total chunks = 517
C2: chunk size = 300, overlap = 30, total chunks = 569
C3: chunk size = 300, overlap = 60, total chunks = 633
C4: chunk size = 500, overlap = 0, total chunks = 323
C5: chunk size = 500, overlap = 50, total chunks = 359
C6: chunk size = 500, overlap = 100, total chunks = 398
C7: chunk size = 700, overlap = 0, total chunks = 242
C8: chunk size = 700, overlap = 70, total chunks = 269
C9: chunk size = 700, overlap = 140, total chunks = 295
C10: chunk size = 1000, overlap = 0, total chunks = 180
C11: chunk size = 1000, overlap = 100, total chunks = 199
C12: chunk size = 1000, overlap = 200, total chunks = 217

Active baseline configuration: C12
Active baseline chunks: 217


### 9. Review the chunking results

Once the chunks have been created, this cell prints the total number produced and verifies that the collection is not empty. It then calculates their character lengths and reports basic observations about the resulting distribution. These values show whether the selected settings behaved as expected across pages of different lengths.

This review is a practical checkpoint before embedding begins. If the chunk count or lengths look unusual, I can return to the extraction or parameter cells without spending time encoding an unsuitable collection. It also provides evidence of the chunking configuration used during the experiment.


In [10]:
# ============================================================
# 9. DISPLAY CHUNKING INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)






CHUNKING RESULTS
Number of chunks: 217
Average characters per chunk: 787.47
Shortest chunk: 2 characters
Longest chunk: 1000 characters


### 10. Export the chunks for manual inspection

This section creates a detailed record of the chunking output so that the passages can be examined outside the notebook. It first converts every chunk dictionary into a row containing the chunk identifier, source document, PDF page, character count, chunk settings, and complete text. The rows are collected in a pandas DataFrame, which provides a convenient structure for writing them to Excel.

The output filename is derived from the uploaded PDF name. The workbook contains the chunk data and a separate summary of the configuration, allowing the passages to remain connected to the experiment that produced them. This matters when different chunk sizes or overlap values are tested, since otherwise several exported files could be difficult to compare.

After pandas writes the workbook, openpyxl reopens it to improve readability. The code formats the headers, adjusts alignment, enables wrapping for long text, and assigns more appropriate column widths. These details do not affect retrieval, but they make it much easier to inspect whether a pay rule was split at an awkward location or mixed with a neighboring provision.

The final `files.download(EXCEL_FILE)` call asks the browser to download the workbook automatically. I noticed that Colab may continue this browser-side download even after the next component has started. That behavior does not mean indexing is running again, it only means the download request created in this cell is still being completed.


In [11]:
# ============================================================
# 10. EXPORT GENERATED CHUNKS TO EXCEL
# ============================================================

print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Create an output filename based on the source PDF.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)






EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition_chunks_inspection.xlsx
Chunks exported: 217
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            3              136
        1            4              745
        2            5             1000
        3            5             1000
        4            5              519
        5            6              999
        6            6             1000
        7            6             1000
        8            6             1000
        9            6              681


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 11. Display a small chunk sample

The first three chunks are printed with their identifiers, page numbers, character counts, and text. This gives an immediate view of the extraction and chunking quality without requiring the Excel workbook to be opened. A small sample is usually enough to notice broken spacing, missing text, or unexpectedly short passages.

Only three chunks are displayed to keep the Colab output readable. The complete collection is still available in the Excel export when a more careful review is needed.


In [12]:
# ============================================================
# 11. DISPLAY SAMPLE CHUNKS
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])






SAMPLE CHUNKS

Chunk ID : 0
Page     : 3
Characters: 136
----------------------------------------------------------------------
HANDBOOK ON WORKERS’ STATUTORY MONETARY BENEFITS DEPARTMENT OF LABOR AND EMPLOYMENT BUREAU OF WORKING CONDITIONS INTRAMUROS, MANILA 2024

Chunk ID : 1
Page     : 4
Characters: 745
----------------------------------------------------------------------
The Project Team Atty. Alvin B. Curada, Atty. Carlo S. Casabar, Jennifer I. Obien, Pauli Andrenette Grace D. Ong, and other BWC staff Special Credit: Nicanor V. Bon and Emila T. de Guzman Cover Art: Euniz Johanne G. Bingil All Rights Reserved This Handbook, or any part thereof, may not be reproduced for commercial purposes without the written permission of the Bureau of Working Conditions, Department of Labor and Employment. This Handbook is published by the: Bureau of Working Conditions Department of Labor and Employment 3rd Floor, DOLE Building, Muralla cor. Gen. Luna Streets, Intramuros, Manila Tel. No. (02) 

### 12. Load the embedding model

This cell initializes `SentenceTransformer` using the name stored earlier in `EMBEDDING_MODEL_NAME`. The resulting `embedding_model` converts text into numerical vectors that represent semantic meaning rather than relying only on exact matching words.

The model object remains in memory and is reused later for user questions. Reusing the same encoder is important because the question vectors must be directly comparable with the document vectors stored in FAISS.


In [13]:
# ============================================================
# 12. LOAD EMBEDDING MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)






LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: all-MiniLM-L6-v2


### 13. Prepare the text for embedding

The chunk dictionaries include identifiers and page metadata, but `SentenceTransformer.encode()` only needs the passage text. This list comprehension extracts every `chunk["text"]` value into `chunk_texts` while preserving the original order.

That ordering creates an important connection between the embedding array and the `chunks` list. Vector position zero must still refer to chunk zero, and the same relationship must continue throughout the collection so that retrieved positions can be translated back into readable sources.


In [14]:
# ============================================================
# 13. EXTRACT CHUNK TEXT
# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]




### 14. Encode the document chunks

The embedding model processes `chunk_texts` and returns a NumPy array containing one vector for every passage. `normalize_embeddings=True` scales each vector to unit length. This allows the inner-product index used in the next cell to produce scores that behave like cosine similarity.

The vectors are converted to `float32` because that is the numeric type expected by FAISS. The printed array shape offers a useful check: its first value should match the number of chunks, while the second value represents the embedding dimension produced by MiniLM.


In [15]:
# ============================================================
# 14. CREATE DOCUMENT EMBEDDINGS
# ============================================================

print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)






CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embedding matrix shape:
(217, 384)


### 15. Build the FAISS index

This cell reads the vector dimension from the embedding array and uses it to create `faiss.IndexFlatIP`. The index performs exact inner-product comparisons rather than using an approximate search structure. Since the vectors were normalized, a larger inner product indicates greater semantic similarity between the question and a chunk.

Calling `index.add(embeddings)` stores every document vector in its current order. The reported `index.ntotal` should match the chunk count. This confirmation tells me that no passage was lost between chunk creation and vector indexing.


In [16]:
# ============================================================
# 15. CREATE FAISS VECTOR STORE
# ============================================================

print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)






CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 217


### 15A. Build the experimental FAISS indexes

This section prepares a separate vector store for every chunking configuration. Each configuration uses the same extracted PDF pages and embedding model, while only the chunk size and overlap are changed. This keeps the experiment controlled and allows the retrieval results to be compared fairly.

For each configuration, the corresponding chunk texts are encoded into normalized embeddings. A new `IndexFlatIP` FAISS index is then created and filled with those vectors. Creating a new index matters because vectors generated from one chunking configuration must not be mixed with chunks from another configuration.

The resulting chunks, FAISS index, and processing time are stored together in `experiment_vector_stores`. Keeping them under their configuration identifiers makes it possible to run the same questions against C1 through C12 without manually rebuilding the notebook for every test. The elapsed time is also recorded because computational efficiency is one of the factors considered in the activity.

In [17]:
# ============================================================
# 15A. BUILD A FAISS INDEX FOR EVERY CONFIGURATION
# ============================================================

import time

experiment_vector_stores = {}

print("\n")
print("=" * 70)
print("BUILDING EXPERIMENTAL FAISS INDEXES")
print("=" * 70)

for configuration in CHUNK_CONFIGURATIONS:

    config_id = configuration["config"]
    chunk_size = configuration["chunk_size"]
    chunk_overlap = configuration["chunk_overlap"]

    configuration_chunks = chunks_by_configuration[config_id]

    configuration_texts = [
        chunk["text"]
        for chunk in configuration_chunks
    ]

    start_time = time.perf_counter()

    configuration_embeddings = embedding_model.encode(
        configuration_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    configuration_embeddings = configuration_embeddings.astype(
        "float32"
    )

    configuration_index = faiss.IndexFlatIP(
        configuration_embeddings.shape[1]
    )

    configuration_index.add(
        configuration_embeddings
    )

    elapsed_time = time.perf_counter() - start_time

    experiment_vector_stores[config_id] = {
        "config": config_id,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "chunks": configuration_chunks,
        "index": configuration_index,
        "total_chunks": len(configuration_chunks),
        "processing_time_seconds": elapsed_time
    }

    print(
        f"{config_id}: "
        f"{len(configuration_chunks)} chunks, "
        f"{configuration_index.ntotal} vectors, "
        f"{elapsed_time:.2f} seconds"
    )


# Keep the original notebook connected to the C12 baseline.
chunks = experiment_vector_stores["C12"]["chunks"]
index = experiment_vector_stores["C12"]["index"]

print("\nAll 12 FAISS indexes were created successfully.")
print("Active notebook configuration: C12")



BUILDING EXPERIMENTAL FAISS INDEXES
C1: 517 chunks, 517 vectors, 0.73 seconds
C2: 569 chunks, 569 vectors, 1.03 seconds
C3: 633 chunks, 633 vectors, 0.91 seconds
C4: 323 chunks, 323 vectors, 0.73 seconds
C5: 359 chunks, 359 vectors, 0.76 seconds
C6: 398 chunks, 398 vectors, 0.79 seconds
C7: 242 chunks, 242 vectors, 0.52 seconds
C8: 269 chunks, 269 vectors, 0.80 seconds
C9: 295 chunks, 295 vectors, 1.51 seconds
C10: 180 chunks, 180 vectors, 0.90 seconds
C11: 199 chunks, 199 vectors, 1.22 seconds
C12: 217 chunks, 217 vectors, 1.19 seconds

All 12 FAISS indexes were created successfully.
Active notebook configuration: C12


### 15B. Define the Fixed Test Questions and Ground Truth

This section records the nine questions used throughout the chunking experiment. The question set contains three easy, three average, and three difficult questions. Keeping them in one structured list ensures that every configuration receives exactly the same retrieval test, which makes the comparison fair and consistent.

Each question includes its expected answer, source reference, and evidence-unit identifiers. These details serve as the ground truth when reviewing whether the top-three retrieved chunks contain enough evidence to answer the question. Although chunk identifiers may change when the chunk size and overlap are adjusted, the required supporting information remains the same.

In [18]:
# ============================================================
# 15B. DEFINE TEST QUESTIONS AND GROUND TRUTH
# ============================================================

TEST_QUESTIONS = [
    {
        "question_id": "E1",
        "difficulty": "Easy",
        "question": (
            "I worked for eight hours on a regular holiday. "
            "It was not my rest day, and I did not work overtime. "
            "What percentage of my basic wage should I receive?"
        ),
        "expected_answer": (
            "The worker should receive 200% of the basic wage "
            "for the first eight hours."
        ),
        "source_reference": (
            "PDF p. 28, Holiday Pay, "
            "D. Holiday Pay Rates, item 2"
        ),
        "evidence_units": ["HP-1"]
    },
    {
        "question_id": "E2",
        "difficulty": "Easy",
        "question": (
            "I have worked for my employer for at least one year, "
            "and I am covered by the service incentive leave benefit. "
            "How many paid leave days should I receive?"
        ),
        "expected_answer": (
            "The worker should receive five days of paid "
            "service incentive leave."
        ),
        "source_reference": (
            "PDF p. 38, Service Incentive Leave, A. Definition"
        ),
        "evidence_units": ["SIL-1"]
    },
    {
        "question_id": "E3",
        "difficulty": "Easy",
        "question": (
            "How should my minimum 13th-month pay be calculated "
            "from the basic salary I earned during the year?"
        ),
        "expected_answer": (
            "The minimum 13th-month pay is one-twelfth of the "
            "total basic salary earned during the calendar year."
        ),
        "source_reference": (
            "PDF p. 51, Thirteenth-Month Pay, "
            "C. Amount and D. Computation"
        ),
        "evidence_units": ["TM-1"]
    },
    {
        "question_id": "A1",
        "difficulty": "Average",
        "question": (
            "I worked for eight hours on a special non-working day, "
            "and it was also my scheduled rest day. "
            "What additional compensation and total pay percentage "
            "should I receive?"
        ),
        "expected_answer": (
            "The worker receives an additional 50% of the basic wage, "
            "resulting in total pay of 150% for the first eight hours."
        ),
        "source_reference": (
            "PDF p. 32, Premium Pay, "
            "D. Premium Pay Rates, item 3"
        ),
        "evidence_units": ["PP-1"]
    },
    {
        "question_id": "A2",
        "difficulty": "Average",
        "question": (
            "I resigned in September before the regular payment of "
            "the 13th-month benefit. Am I still entitled to it, "
            "and how should it be calculated?"
        ),
        "expected_answer": (
            "Yes. The worker is entitled to proportionate 13th-month "
            "pay equal to one-twelfth of the total basic salary earned "
            "up to the resignation date."
        ),
        "source_reference": (
            "PDF pp. 52-53, Thirteenth-Month Pay, "
            "G. Resigned or Separated Employee"
        ),
        "evidence_units": ["TM-2", "TM-3"]
    },
    {
        "question_id": "A3",
        "difficulty": "Average",
        "question": (
            "I did not use all of my service incentive leave, "
            "and I am resigning. Can the unused leave be paid in cash, "
            "and which salary rate should be used?"
        ),
        "expected_answer": (
            "Yes. Unused service incentive leave may be converted "
            "to cash upon resignation using the salary rate on the "
            "date of commutation."
        ),
        "source_reference": (
            "PDF p. 39, Service Incentive Leave, D. Conversion"
        ),
        "evidence_units": ["SIL-2", "SIL-3"]
    },
    {
        "question_id": "D1",
        "difficulty": "Difficult",
        "question": (
            "I worked for 10 hours on a regular holiday that also "
            "fell on my scheduled rest day. What rate applies to my "
            "first eight hours, what rate applies to my two overtime "
            "hours, and what is my total pay as a percentage of my "
            "basic daily wage? Assume my basic daily wage covers "
            "eight hours."
        ),
        "expected_answer": (
            "The first eight hours receive 260% of the basic daily "
            "wage. Each overtime hour receives 338% of the basic "
            "hourly rate. The total for ten hours is 344.5% of the "
            "basic daily wage."
        ),
        "source_reference": (
            "PDF pp. 32-33, Premium Pay item 4 and "
            "Overtime Pay item 5"
        ),
        "evidence_units": ["HP-2", "OT-1"]
    },
    {
        "question_id": "D2",
        "difficulty": "Difficult",
        "question": (
            "My company is closing for reasons that are not due to "
            "serious business losses. I worked for 6 years and 7 months. "
            "My latest monthly salary is PHP 20,000, and I regularly "
            "receive a PHP 2,000 monthly allowance. What minimum "
            "separation pay should I receive?"
        ),
        "expected_answer": (
            "The service period is counted as seven years. The salary "
            "base is PHP 22,000. The minimum separation pay is "
            "PHP 22,000 multiplied by one-half multiplied by seven, "
            "which equals PHP 77,000."
        ),
        "source_reference": (
            "PDF pp. 55-56, Separation Pay, "
            "C. Amount and D. Computation"
        ),
        "evidence_units": ["SP-1", "SP-2"]
    },
    {
        "question_id": "D3",
        "difficulty": "Difficult",
        "question": (
            "I am 60 years old, have worked for a covered establishment "
            "for 14 years and 7 months, and my latest daily rate is "
            "PHP 700. If there is no more favorable retirement plan, "
            "what is my minimum retirement pay?"
        ),
        "expected_answer": (
            "The service period is counted as 15 years. The minimum "
            "retirement pay is PHP 700 multiplied by 22.5 days "
            "multiplied by 15 years, which equals PHP 236,250."
        ),
        "source_reference": (
            "PDF p. 58, Retirement Pay, "
            "B. Coverage, C. Amount, and D. Computation"
        ),
        "evidence_units": ["RP-1", "RP-2", "RP-3"]
    }
]

print("Total test questions:", len(TEST_QUESTIONS))

for difficulty in ["Easy", "Average", "Difficult"]:

    difficulty_count = sum(
        1
        for item in TEST_QUESTIONS
        if item["difficulty"] == difficulty
    )

    print(f"{difficulty}: {difficulty_count}")

Total test questions: 9
Easy: 3
Average: 3
Difficult: 3


### 15C. Run the Retrieval Experiment

This section runs the same nine questions against each of the twelve FAISS indexes. The questions are encoded once using the same embedding model used for the document chunks. Reusing the question embeddings keeps the experiment efficient while ensuring that the query representation remains identical across all configurations.

For each configuration, FAISS retrieves the three chunks with the highest similarity scores. The retrieved position is matched with the corresponding chunk from that configuration so that its chunk identifier, PDF page, similarity score, and complete text can be recorded.

The resulting retrieval log also includes the question difficulty, expected answer, expected source, and required evidence units. The `Relevant` and `Evidence_Covered` fields are intentionally left blank because relevance must be judged by checking whether the retrieved text genuinely supports the answer. A related keyword or topic alone is not enough to mark a chunk as relevant.

In [19]:
# ============================================================
# 15C. RUN THE RETRIEVAL EXPERIMENT
# ============================================================

print("\n")
print("=" * 70)
print("RUNNING RETRIEVAL EXPERIMENT")
print("=" * 70)

# Encode the nine questions once so every configuration
# receives exactly the same query vectors.
question_texts = [
    item["question"]
    for item in TEST_QUESTIONS
]

question_embeddings = embedding_model.encode(
    question_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

question_embeddings = question_embeddings.astype(
    "float32"
)

retrieval_records = []

for configuration in CHUNK_CONFIGURATIONS:

    config_id = configuration["config"]
    configuration_store = experiment_vector_stores[config_id]

    configuration_index = configuration_store["index"]
    configuration_chunks = configuration_store["chunks"]

    search_k = min(
        TOP_K,
        configuration_index.ntotal
    )

    scores, indices = configuration_index.search(
        question_embeddings,
        search_k
    )

    for question_position, question_data in enumerate(TEST_QUESTIONS):

        for result_position in range(search_k):

            chunk_position = int(
                indices[question_position][result_position]
            )

            if chunk_position == -1:
                continue

            retrieved_chunk = configuration_chunks[
                chunk_position
            ]

            retrieval_records.append({
                "Configuration": config_id,
                "Chunk_Size": configuration["chunk_size"],
                "Chunk_Overlap": configuration["chunk_overlap"],
                "Total_Chunks": configuration_store["total_chunks"],
                "Indexing_Time_Seconds": round(
                    configuration_store[
                        "processing_time_seconds"
                    ],
                    4
                ),
                "Question_ID": question_data["question_id"],
                "Difficulty": question_data["difficulty"],
                "Question": question_data["question"],
                "Expected_Answer": question_data["expected_answer"],
                "Expected_Source": question_data["source_reference"],
                "Expected_Evidence": ", ".join(
                    question_data["evidence_units"]
                ),
                "Retrieved_Rank": result_position + 1,
                "Retrieved_Chunk_ID": retrieved_chunk["chunk_id"],
                "Retrieved_Page": retrieved_chunk["page"],
                "Similarity_Score": float(
                    scores[question_position][result_position]
                ),
                "Retrieved_Chunk_Text": retrieved_chunk["text"],

                # These two fields will be completed after
                # manually reviewing the retrieved evidence.
                "Relevant": "",
                "Evidence_Covered": ""
            })

    print(
        f"{config_id}: "
        f"{len(TEST_QUESTIONS)} questions completed, "
        f"top {search_k} retrieved"
    )


retrieval_log_df = pd.DataFrame(
    retrieval_records
)

expected_rows = (
    len(CHUNK_CONFIGURATIONS)
    * len(TEST_QUESTIONS)
    * TOP_K
)

print("\n")
print("=" * 70)
print("RETRIEVAL EXPERIMENT SUMMARY")
print("=" * 70)

print("Configurations tested :", len(CHUNK_CONFIGURATIONS))
print("Questions tested      :", len(TEST_QUESTIONS))
print("Top-K                 :", TOP_K)
print("Expected records      :", expected_rows)
print("Actual records        :", len(retrieval_log_df))

if len(retrieval_log_df) == expected_rows:
    print("Status                : COMPLETE")
else:
    print("Status                : CHECK THE RETRIEVAL RESULTS")



RUNNING RETRIEVAL EXPERIMENT
C1: 9 questions completed, top 3 retrieved
C2: 9 questions completed, top 3 retrieved
C3: 9 questions completed, top 3 retrieved
C4: 9 questions completed, top 3 retrieved
C5: 9 questions completed, top 3 retrieved
C6: 9 questions completed, top 3 retrieved
C7: 9 questions completed, top 3 retrieved
C8: 9 questions completed, top 3 retrieved
C9: 9 questions completed, top 3 retrieved
C10: 9 questions completed, top 3 retrieved
C11: 9 questions completed, top 3 retrieved
C12: 9 questions completed, top 3 retrieved


RETRIEVAL EXPERIMENT SUMMARY
Configurations tested : 12
Questions tested      : 9
Top-K                 : 3
Expected records      : 324
Actual records        : 324
Status                : COMPLETE


### 15D. Export the Experimental Retrieval Log

This section saves the chunking experiment results in a structured Excel workbook. The detailed retrieval log contains one row for every retrieved chunk, resulting in 324 records from twelve configurations, nine questions, and three retrieval ranks.

Separate worksheets are included for the configuration settings, question ground truth, performance by difficulty, and detailed retrieval records. The metric columns remain empty at this stage because they can only be calculated after the retrieved chunks have been reviewed for relevance.

The `Relevant` and `Evidence_Covered` columns in the detailed log are intended for manual evaluation. A chunk should be marked relevant only when its text contains evidence that genuinely supports the expected answer. This review creates the labels needed to calculate Precision@3, Recall@3, and MRR@3 consistently.

In [20]:
# ============================================================
# 15D. EXPORT THE EXPERIMENTAL RETRIEVAL LOG
# ============================================================

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

EXPERIMENT_LOG_FILE = (
    "FAISS_Chunking_Experiment_Retrieval_Log.xlsx"
)


# ------------------------------------------------------------
# PREPARE THE CONFIGURATION SUMMARY
# ------------------------------------------------------------

configuration_rows = []

for configuration in CHUNK_CONFIGURATIONS:

    config_id = configuration["config"]
    configuration_store = experiment_vector_stores[config_id]

    configuration_rows.append({
        "Configuration": config_id,
        "Chunk_Size": configuration["chunk_size"],
        "Chunk_Overlap": configuration["chunk_overlap"],
        "Total_Chunks": configuration_store["total_chunks"],
        "Indexing_Time_Seconds": round(
            configuration_store[
                "processing_time_seconds"
            ],
            4
        ),
        "Precision_at_3": "",
        "Recall_at_3": "",
        "MRR_at_3": ""
    })

configuration_summary_df = pd.DataFrame(
    configuration_rows
)


# ------------------------------------------------------------
# PREPARE THE DIFFICULTY SUMMARY
# ------------------------------------------------------------

difficulty_rows = []

for configuration in CHUNK_CONFIGURATIONS:

    for difficulty in ["Easy", "Average", "Difficult"]:

        difficulty_rows.append({
            "Configuration": configuration["config"],
            "Difficulty": difficulty,
            "Precision_at_3": "",
            "Recall_at_3": "",
            "MRR_at_3": ""
        })

difficulty_summary_df = pd.DataFrame(
    difficulty_rows
)


# ------------------------------------------------------------
# PREPARE THE GROUND-TRUTH TABLE
# ------------------------------------------------------------

ground_truth_rows = []

for question_data in TEST_QUESTIONS:

    ground_truth_rows.append({
        "Question_ID": question_data["question_id"],
        "Difficulty": question_data["difficulty"],
        "Question": question_data["question"],
        "Expected_Answer": question_data["expected_answer"],
        "Expected_Source": question_data["source_reference"],
        "Expected_Evidence": ", ".join(
            question_data["evidence_units"]
        )
    })

ground_truth_df = pd.DataFrame(
    ground_truth_rows
)


# ------------------------------------------------------------
# WRITE THE WORKBOOK
# ------------------------------------------------------------

with pd.ExcelWriter(
    EXPERIMENT_LOG_FILE,
    engine="openpyxl"
) as writer:

    configuration_summary_df.to_excel(
        writer,
        sheet_name="Configuration_Results",
        index=False
    )

    difficulty_summary_df.to_excel(
        writer,
        sheet_name="Difficulty_Performance",
        index=False
    )

    ground_truth_df.to_excel(
        writer,
        sheet_name="Ground_Truth",
        index=False
    )

    retrieval_log_df.to_excel(
        writer,
        sheet_name="Detailed_Retrieval_Log",
        index=False
    )


# ------------------------------------------------------------
# FORMAT THE WORKBOOK
# ------------------------------------------------------------

workbook = load_workbook(
    EXPERIMENT_LOG_FILE
)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)

header_font = Font(
    color="FFFFFF",
    bold=True
)

for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions

    for cell in worksheet[1]:

        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    for row in worksheet.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    # Assign a readable default width.
    for column_number in range(
        1,
        worksheet.max_column + 1
    ):

        worksheet.column_dimensions[
            get_column_letter(column_number)
        ].width = 18


# Adjust the longer text columns.
ground_truth_sheet = workbook["Ground_Truth"]

ground_truth_sheet.column_dimensions["C"].width = 60
ground_truth_sheet.column_dimensions["D"].width = 60
ground_truth_sheet.column_dimensions["E"].width = 48
ground_truth_sheet.column_dimensions["F"].width = 30


retrieval_sheet = workbook["Detailed_Retrieval_Log"]

retrieval_sheet.column_dimensions["H"].width = 60
retrieval_sheet.column_dimensions["I"].width = 60
retrieval_sheet.column_dimensions["J"].width = 48
retrieval_sheet.column_dimensions["K"].width = 25
retrieval_sheet.column_dimensions["P"].width = 90
retrieval_sheet.column_dimensions["Q"].width = 14
retrieval_sheet.column_dimensions["R"].width = 25


# Give the detailed records enough height to preview their text.
for row_number in range(
    2,
    retrieval_sheet.max_row + 1
):
    retrieval_sheet.row_dimensions[
        row_number
    ].height = 60


workbook.save(
    EXPERIMENT_LOG_FILE
)


# ------------------------------------------------------------
# VERIFY AND DOWNLOAD
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("EXPERIMENT LOG CREATED")
print("=" * 70)

print("File:", EXPERIMENT_LOG_FILE)
print(
    "Configuration rows:",
    len(configuration_summary_df)
)
print(
    "Difficulty rows:",
    len(difficulty_summary_df)
)
print(
    "Ground-truth questions:",
    len(ground_truth_df)
)
print(
    "Detailed retrieval rows:",
    len(retrieval_log_df)
)

files.download(
    EXPERIMENT_LOG_FILE
)



EXPERIMENT LOG CREATED
File: FAISS_Chunking_Experiment_Retrieval_Log.xlsx
Configuration rows: 12
Difficulty rows: 36
Ground-truth questions: 9
Detailed retrieval rows: 324


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 16. Link vector positions to chunk metadata

FAISS stores vectors by numeric position and returns those positions during a search. The `chunk_lookup` dictionary connects each position to the corresponding entry in `chunks`, including its identifier, page, and original text.

This mapping turns a numerical search result into evidence that can be read and cited. It also depends on the order preserved during text extraction and embedding, which is why the earlier cells avoid rearranging the chunk list.


In [ ]:
# ============================================================
# 16. CREATE CHUNK LOOKUP
# ============================================================

# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}




### 17. Define the basic retrieval function

The `retrieve_documents()` function provides the first complete test of the vector store. It begins by validating the question and `top_k` value so that empty input or an invalid result count is rejected with a clear message. The question is then encoded by the same `embedding_model`, normalized, and converted to `float32` to match the document vectors.

`search_k` prevents the function from requesting more results than the index contains. The call to `index.search()` returns similarity scores and vector positions. Each valid position is translated through `chunk_lookup`, allowing the function to recover the matched passage, page number, and chunk identifier.

The function also considers the matched chunk together with its immediately previous and next positions. A page check prevents these neighboring positions from crossing into a different PDF page, and `seen_chunk_ids` prevents duplicates when neighboring groups overlap. This extra context can help when a sentence or rule continues just beyond the highest-scoring chunk.

Every returned result records both its displayed order and the rank of the center match that introduced it. It also carries the similarity score and full text. I find this structure useful because later cells can display the evidence clearly without repeating the details of FAISS search and metadata lookup.


In [ ]:
# ============================================================
# 17. DEFINE RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []
    seen_chunk_ids = set()

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        center_position = int(idx)
        center_chunk = chunk_lookup[center_position]
        center_page = center_chunk["page"]
        center_score = float(scores[0][rank])

        # Include the preceding chunk before the matched chunk.
        # This helps preserve headings and rule context.
        candidate_positions = [
            center_position - 1,
            center_position
        ]

        for position in candidate_positions:

            if position not in chunk_lookup:
                continue

            chunk = chunk_lookup[position]

            # Never cross into another PDF page.
            if chunk["page"] != center_page:
                continue

            if chunk["chunk_id"] in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk["chunk_id"])

            results.append({
                "rank": len(results) + 1,
                "retrieval_rank": rank + 1,
                "chunk_id": chunk["chunk_id"],
                "page": chunk["page"],
                "score": center_score,
                "text": chunk["text"]
            })

    return results




### 18. Test the vector store

This optional cell allows a question to be entered immediately after the index is created. If the input is not empty, it calls `retrieve_documents()` and prints each result with its chunk identifier, page, similarity score, and text. This gives a direct way to see whether semantic retrieval is pointing toward the expected handbook section.

Pressing Enter skips the test without affecting the saved index. I can therefore use the same notebook either interactively, when I want to inspect retrieval, or more quickly when I only need to prepare the files for Component II.


In [ ]:
# ============================================================
# 18. TEST RETRIEVAL
# ============================================================

print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])






TESTING VECTOR STORE

Enter a test question (or press Enter to skip): 


### 19. Save the FAISS index

The completed FAISS structure is saved as `rag_faiss.index` through `faiss.write_index()`. This file contains the document vectors in a form FAISS can reload later.

Saving the index avoids repeating extraction and embedding whenever the same handbook is reused. The printed filename also confirms where the vector store was written in the current Colab session.


In [ ]:
# ============================================================
# 19. SAVE FAISS INDEX
# ============================================================

FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)





FAISS index saved: rag_faiss.index


### 20. Save the chunks and their metadata

This section stores the `chunks` list in `rag_chunks.pkl` using pickle. The FAISS file alone only knows vector positions, so the saved chunk metadata is necessary to recover the readable passage, source page, and chunk identifier associated with each position.

The index and metadata files should therefore be treated as a pair. Loading one without the other would either remove the search capability or remove the information needed to understand the search results.


In [ ]:
# ============================================================
# 20. SAVE CHUNKS AND METADATA
# ============================================================

CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)




Chunks saved: rag_chunks.pkl


### 21. Save the indexing configuration

The `config` dictionary records the chunk size, overlap, default retrieval count, embedding model, and source filename used for the current index. It is saved as `rag_config.pkl` so that the experiment settings remain available with the vector store.

This small file supports reproducibility. If retrieval behavior changes after another experiment, the saved configuration helps confirm whether the difference came from the document, the embedding model, or the chunking values.


In [ ]:
# ============================================================
# 21. SAVE INDEXING CONFIGURATION
# ============================================================

CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)




Configuration saved: rag_config.pkl


### 22. Confirm that indexing is complete

The last cell in Component I prints a summary of the indexed document, number of chunks, number of stored vectors, selected chunk settings, and embedding model. It also lists the files created during the workflow.

This summary gives the indexing stage a clear ending. Before moving into prompt processing, I can check that the vector count matches the chunk count and that the displayed settings are the ones intended for the experiment.


In [ ]:
# ============================================================
# 22. INDEXING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)




INDEXING COMPLETE
Source document : Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition.pdf
Number of chunks: 217
Vectors stored  : 217
Chunk size      : 1000
Chunk overlap   : 200
Embedding model : all-MiniLM-L6-v2

Saved files:
   Workers-Statutory-Monetary-Benefits-Handbook-2024-Edition_chunks_inspection.xlsx
   rag_faiss.index
   rag_chunks.pkl
   rag_config.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


#Component II: Prompt and Retrieval

### 1. Import the prompt and generation tools

Component II imports the classes needed for language-model inference and reranking. `AutoTokenizer` prepares text for Qwen, while `AutoModelForCausalLM` loads the model that generates the final response. `CrossEncoder` supplies the relevance model used after the initial FAISS search.

PyTorch manages device selection, tensor movement, numerical precision, and generation without gradient calculations. The regular-expression module is used later to remove any thinking tags that might remain in the decoded response. These imports build on the index, chunks, embedding model, and lookup table that are already available from Component I.


In [ ]:

# ============================================================
# RAG PROMPT PROCESSING
# TOKEN-LIMITED CONTEXT, ANSWER GENERATION, AND DIAGNOSTICS
# Model: Qwen/Qwen3-1.7B with generic cross-encoder reranking
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from sentence_transformers import CrossEncoder

import torch
import re




### 2. Select the available execution device

This cell checks `torch.cuda.is_available()` and chooses either CUDA or CPU. When a GPU is available, the model uses `torch.float16` to reduce memory use. The CPU fallback uses `torch.float32`, which is more broadly supported but noticeably slower for generation and reranking.

The selected device and GPU name are printed before the models are loaded. This observation is important in Colab because selecting a T4 runtime in the menu does not always guarantee that the active session is currently connected to it. The warning makes an unintended CPU session visible before a long model-loading step begins.


In [ ]:
# ============================================================
# 2. CHECK DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.float16 if device == "cuda" else torch.float32

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU was detected. CPU fallback will be slow.")




RAG PROMPT PROCESSING
Device: cuda
GPU: Tesla T4


### 3. Load Qwen and the reranking model

The language model is identified as `Qwen/Qwen3-1.7B`. The cell also sets the maximum input length, maximum generated tokens, and a reserved margin that protects the prompt budget. These values define how much retrieved evidence can be shown to Qwen and how long its answer may become.

`AutoTokenizer.from_pretrained()` loads Qwen's tokenizer, and `AutoModelForCausalLM.from_pretrained()` loads the model weights using the precision selected in the previous cell. The model is moved to the active device and placed in evaluation mode because the notebook performs inference rather than training.

The cross-encoder reranker is then loaded on the same device. It has a different responsibility from Qwen: it compares the full question with each candidate passage and assigns a relevance score. Loading both models here makes the later retrieval and generation functions shorter and allows their initialization messages to be reviewed in one place.


In [ ]:
# ============================================================
# 3. LOAD THE LOCAL LLM
# ============================================================

LLM_NAME = "Qwen/Qwen3-1.7B"

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 128

# Reserve space within the input budget for safety.
RESERVED_TOKEN_MARGIN = 64

print("\nLoading language model:")
print(LLM_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    dtype=model_dtype
)

llm = llm.to(device)
llm.eval()

print("Language model loaded.")

RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("\nLoading reranking model:")
print(RERANKER_NAME)

reranker = CrossEncoder(
    RERANKER_NAME,
    device=device
)

print("Reranking model loaded.")





Loading language model:
Qwen/Qwen3-1.7B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (12 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranking model loaded.


### 4. Define token counting and chat formatting

The `count_tokens()` helper measures text with the active Qwen tokenizer. Character counts are not reliable for prompt budgeting because words and punctuation can be divided into tokens in different ways. By calling `tokenizer.encode()` without truncation, the notebook can check the real input length before generation.

`ANSWER_SYSTEM_MESSAGE` defines the model's role as a document-grounded Philippine labor-benefits assistant. It tells Qwen to remain concise, rely only on the supplied handbook evidence, and avoid combining unrelated rules. Keeping this instruction in a named variable makes its purpose clear and allows the same wording to be reused consistently.

The `format_qwen_chat_prompt()` function creates a conversation containing one system message and one user message. `tokenizer.apply_chat_template()` converts that conversation into the exact text format expected by Qwen. Thinking mode is disabled so the model produces a direct answer rather than exposing a longer internal reasoning sequence.

Together, these helpers create a consistent boundary between ordinary text and model-ready input. Later functions can focus on selecting evidence and generating answers without having to repeat the tokenizer-specific formatting details.


In [ ]:
# ============================================================
# 4. DEFINE TOKEN COUNTING FUNCTION
# ============================================================

def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False
        )
    )


ANSWER_SYSTEM_MESSAGE = (
    "You are a document-grounded Philippine labor-benefits "
    "information assistant for workers. Answer concisely in English "
    "using only the supplied handbook evidence. Do not guess, combine "
    "unrelated rules, or use information outside the provided evidence."
)


def format_qwen_chat_prompt(user_content, system_content):
    """Apply Qwen3's chat template with thinking disabled."""

    messages = [
        {
            "role": "system",
            "content": system_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )




### 5. Preserve reranker priority

The `prioritize_documents()` function sorts the retrieved dictionaries according to their `score`, placing the largest cross-encoder value first. A missing score is treated as negative infinity, so incomplete entries naturally fall to the end rather than interrupting the process.

Although the reranked retrieval function already returns ordered results, this additional sort keeps prompt construction defensive. It ensures that the strongest available evidence receives the first opportunity to fit within the token budget.


In [ ]:
# ============================================================
# 5. PRIORITIZE RETRIEVED DOCUMENTS
# ============================================================

def prioritize_documents(question, retrieved_documents):
    """
    Preserve cross-encoder relevance ranking.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )





### 6. Build a prompt within the token budget

The `create_rag_prompt()` function is responsible for turning a question and retrieved passages into the final input sent to Qwen. It first validates the question and confirms that evidence was provided. The retrieved documents are then passed through `prioritize_documents()` so that passages with higher reranker scores are considered before weaker candidates.

The function creates a detailed request that explains how the answer should be grounded. It asks the model to identify the applicable benefit and conditions, distinguish an additional premium from a total rate, use exact figures only when supported, and avoid mixing rules for regular holidays, special days, rest days, ordinary hours, and overtime. It also defines the exact fallback sentence to use when the evidence does not support an answer. These instructions are especially important in this handbook because several neighboring provisions use similar percentages under different conditions.

Each retrieved passage is formatted with its PDF page and chunk identifier. The function tentatively adds one source at a time, builds the complete user message, and applies Qwen's chat template. It then uses `count_tokens()` to measure the actual formatted prompt. If the next source would exceed `MAX_INPUT_TOKENS` after accounting for `RESERVED_TOKEN_MARGIN`, that source is not added.

This incremental approach protects the original question and core instructions instead of allowing evidence to be cut off silently. It also means the highest-ranked passages are most likely to remain in the prompt when space is limited. If even the basic question and instructions exceed the budget, the function raises an error because generating from a truncated request would hide an important problem.

Finally, the function prints diagnostics showing the original question, number of retrieved chunks, number actually included, final prompt length, and input limit. It returns the formatted prompt together with the included source records. These diagnostics make the prompt-building decision visible, which helps when reviewing why a useful passage did or did not reach the language model.


In [ ]:
# ============================================================
# 6. DEFINE THE TOKEN-LIMITED RAG PROMPT
# ============================================================

def create_rag_prompt(
    question,
    retrieved_documents
):
    """
    Build a prompt that preserves the question and includes
    as much relevant retrieved evidence as the input budget
    allows.

    Context is trimmed only at token boundaries.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    instruction = (
    "Answer the worker's question using only the supplied handbook "
    "evidence. Identify the exact labor benefit and the conditions "
    "that apply to the worker's situation. Clearly distinguish a "
    "total pay rate from an additional premium. Give the exact amount, "
    "percentage, or formula only when it is directly supported by the "
    "evidence. Do not combine rules for different worker categories, "
    "benefits, regular holidays, special days, rest days, ordinary "
    "working hours, or overtime. If the retrieved evidence describes "
    "a different situation from the question, do not use that rule. "
    "Answer directly and concisely in English. Do not include a PDF "
    "page citation because the application adds the page number from "
    "the highest-ranked source. Do not guess or use outside knowledge. "
    "If the evidence does not directly support an answer, respond "
    "exactly: 'I cannot find the answer in the provided document.'"
)

    # Keep the question near the beginning.
    prompt_prefix = (
        f"Question: {question}\n\n"
        f"{instruction}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = "\n\nAnswer:"

    # Tokenize the fixed parts first.
    prefix_tokens = tokenizer.encode(
        prompt_prefix,
        add_special_tokens=False
    )

    suffix_tokens = tokenizer.encode(
        prompt_suffix,
        add_special_tokens=False
    )

    # Reserve tokens for Qwen's system/user chat markers.
    chat_wrapper_tokens = count_tokens(
        format_qwen_chat_prompt("", ANSWER_SYSTEM_MESSAGE)
    )

    # Keep the final chat-formatted prompt within the limit.
    available_context_tokens = (
        MAX_INPUT_TOKENS
        - len(prefix_tokens)
        - len(suffix_tokens)
        - chat_wrapper_tokens
        - RESERVED_TOKEN_MARGIN
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room "
            "for retrieved evidence. Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:

        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        # Tokenize each source separately.
        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode(
                "\n\n",
                add_special_tokens=False
            )
            if selected_parts
            else []
        )

        tokens_available = (
            remaining_tokens - len(separator_tokens)
        )

        if tokens_available <= 0:
            break

        # Include the full chunk when it fits.
        # Otherwise, retain the beginning of the chunk.
        included_tokens = source_tokens[
            :tokens_available
        ]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens)
            + len(included_tokens)
        )

        # Stop if this chunk had to be truncated.
        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    rag_content = (
        prompt_prefix
        + context
        + prompt_suffix
    )

    # Apply Qwen's chat template before final token counting.
    prompt = format_qwen_chat_prompt(
        rag_content,
        ANSWER_SYSTEM_MESSAGE
    )

    # Final check using the actual Qwen tokenizer.
    final_token_count = count_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, "
            f"exceeding the {MAX_INPUT_TOKENS}-token limit. "
            "Reduce the context budget."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Final prompt tokens:", final_token_count)
    print("Input token limit:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")

    for i, part in enumerate(selected_parts, start=1):
      print(f"\nEvidence {i}:")
      print(part)

    return prompt




### 7. Generate an answer safely

The `generate_answer()` function receives the already formatted Qwen prompt and converts it into PyTorch tensors. Tokenization is performed without truncation because the previous function has already managed the input budget. The tensors are moved to the same device as the language model so generation does not fail because of a CPU and GPU mismatch.

The actual input length is checked again before inference. This second validation may appear cautious, but it protects the model if the function is called with a prompt that did not come from `create_rag_prompt()`. The cell prints the measured input and configured limit, making the final size visible immediately before generation.

Inside `torch.no_grad()`, `llm.generate()` produces up to `MAX_NEW_TOKENS` new tokens. Disabling gradients reduces memory use because the notebook is not training the model. Only the tokens appearing after the original prompt are selected, which prevents the decoded result from repeating the full system message, question, and evidence.

The new tokens are decoded into text, and regular expressions remove any remaining `<think>` block or tag. The final `.strip()` returns a clean answer. This cleanup supports the decision to present concise worker-facing output while still allowing Qwen to use its normal generation interface.


In [ ]:
# ============================================================
# 7. DEFINE ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    """
    Generate an answer without silently truncating the prompt.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual input tokens:", input_length)
    print("Maximum input tokens:", MAX_INPUT_TOKENS)

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Input contains {input_length} tokens, "
            f"but the model limit is {MAX_INPUT_TOKENS}."
        )

    # Move model inputs to the selected device.
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate deterministically.
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Causal models return the prompt plus newly generated tokens.
    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer




### 8. Retrieve candidates and rerank them

The `retrieve_documents_reranked()` function combines fast vector search with a more careful relevance check. It first validates the question, requested top count, and candidate-pool size. The pool must be at least as large as `top_k`, since reranking cannot return more final passages than it receives from FAISS.

The question is encoded through the same normalized `embedding_model` used in Component I and converted to `float32`. FAISS then searches for up to 30 candidates, or fewer when the index contains fewer vectors. This first stage is efficient and broad, but semantic similarity alone may place a related yet inapplicable labor rule near the top.

For every valid FAISS position, the function recovers the passage through `chunk_lookup` and records the original similarity score and FAISS rank. It then creates question-passage pairs and sends them to the `reranker`. The cross-encoder reads both texts together, allowing it to consider conditions such as whether the question refers to a rest day, the first eight hours, or overtime.

The candidate dictionaries receive their reranker scores and are sorted from highest to lowest. Only the requested top results are returned. Each result still carries its FAISS score and rank, so the notebook can show how the second-stage ordering differs from the initial vector search.

The diagnostic output lists the candidate-pool size and the final ranked pages and chunks. I use this information to separate retrieval problems from generation problems. If the correct rule never appears among the selected passages, retrieval needs attention. If the correct passage is present but the answer is wrong, the difficulty lies in evidence interpretation or calculation.


In [ ]:
# ============================================================
# 8. DEFINE GENERIC RERANKED RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents_reranked(
    query,
    top_k=3,
    candidate_pool_size=30
):
    """Retrieve with FAISS, then rerank with a cross-encoder."""

    if not query or not query.strip():
        raise ValueError("The retrieval query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    if index.ntotal == 0:
        return []

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    candidate_k = min(
        max(candidate_pool_size, top_k),
        index.ntotal
    )

    faiss_scores, faiss_indices = index.search(
        query_embedding,
        candidate_k
    )

    candidates = []

    for faiss_rank, idx in enumerate(faiss_indices[0]):

        if idx == -1:
            continue

        position = int(idx)
        chunk = chunk_lookup[position]

        candidates.append({
            "position": position,
            "faiss_rank": faiss_rank + 1,
            "faiss_score": float(faiss_scores[0][faiss_rank]),
            "chunk": chunk
        })

    if not candidates:
        return []

    query_document_pairs = [
        [query, candidate["chunk"]["text"]]
        for candidate in candidates
    ]

    reranker_scores = reranker.predict(
        query_document_pairs,
        batch_size=16,
        show_progress_bar=False
    )

    for candidate, reranker_score in zip(
        candidates,
        reranker_scores
    ):
        candidate["reranker_score"] = float(reranker_score)

    candidates.sort(
        key=lambda candidate: candidate["reranker_score"],
        reverse=True
    )

    selected_candidates = candidates[:top_k]

    print("\n--- RERANKING DIAGNOSTICS ---")
    print("FAISS candidates:", len(candidates))

    results = []

    for rank, candidate in enumerate(selected_candidates, start=1):

        chunk = candidate["chunk"]

        print(
            f"Rank {rank}: Page {chunk['page']} | "
            f"Chunk {chunk['chunk_id']} | "
            f"Reranker {candidate['reranker_score']:.4f} | "
            f"FAISS rank {candidate['faiss_rank']}"
        )

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": candidate["reranker_score"],
            "faiss_score": candidate["faiss_score"],
            "faiss_rank": candidate["faiss_rank"],
            "text": chunk["text"]
        })

    return results


def attach_source_citation(answer, retrieved_documents):
    """Append the page from the highest-ranked source."""

    fallback = "I cannot find the answer in the provided document."

    if not answer or not retrieved_documents:
        return answer

    if answer.strip() == fallback:
        return answer.strip()

    cleaned_answer = re.sub(
        r"\s*\(PDF page\s+\d+\)\.?\s*$",
        "",
        answer.strip(),
        flags=re.IGNORECASE
    ).rstrip(" .")

    source_page = retrieved_documents[0]["page"]

    return f"{cleaned_answer}. (PDF page {source_page})"




### 9. Connect retrieval, prompting, and generation

The `rag_query()` function joins the major operations into one reusable workflow. It begins by calling `retrieve_documents_reranked()` with the original English question. No query rewriting is performed, so the retrieval diagnostics reflect the exact wording entered by the user.

The reranked results are passed to `create_rag_prompt()`, which selects the passages that fit and returns the formatted Qwen input. `generate_answer()` then produces the document-grounded response. Keeping these calls in one function reduces repetition in the interactive loop while preserving separate helpers that can still be tested individually.

After generation, the function obtains the page of the highest-ranked retrieved source and appends it to the answer. It also builds a diagnostics dictionary containing the original question, retrieval query, source count, included count, and prompt token information. These details support later review of the output.

The function returns four values: the final answer, the retrieved sources, the complete prompt, and the diagnostics. This design allows the user interface to show a concise answer first while still keeping the evidence and prompt available through separate commands.


In [ ]:
# ============================================================
# 9. DEFINE COMPLETE RAG QUERY FUNCTION
# ============================================================


def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve FAISS candidates using the English question.
    2. Rerank candidates with the cross-encoder.
    3. Build a prompt using the unchanged question.
    4. Generate an answer and attach a verified source page.
    """

    # STEP 1: Validate the question.
    question = question.strip()

    if not question:
        return (
            "Please enter a valid question.",
            [],
            ""
        )

    # STEP 2: Use the original English question for retrieval.
    retrieval_query = question

    print("\n--- RETRIEVAL QUERY ---")
    print(retrieval_query)

    # STEP 3: Retrieve and rerank relevant document chunks.
    retrieved_documents = retrieve_documents_reranked(
        retrieval_query,
        top_k=top_k
    )

    # STEP 4: Handle cases where no evidence is retrieved.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # STEP 5: Use the original English question in the final prompt.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # STEP 6: Generate an answer from the retrieved evidence.
    answer = generate_answer(prompt)

    # STEP 7: Handle an empty model response.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # STEP 8: Attach the highest-ranked source page.
    answer = attach_source_citation(
        answer,
        retrieved_documents
    )

    # STEP 9: Return the results.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )




### 10. Run the interactive question-answering session

This section provides the continuous interface used to test the indexed handbook. It introduces the available commands and initializes variables for the most recent answer, sources, prompt, and diagnostics. Keeping the previous result in memory allows the user to inspect it without repeating a model query.

The `while True` loop accepts a question through `input()` and checks command conditions before starting retrieval. Entering `exit` breaks the loop and ends the session. An empty entry is ignored with a reminder, which prevents unnecessary embedding and model calls.

The `sources` command prints the passages returned by the last retrieval, including their reranker scores and FAISS ranks. The `prompt` command displays the complete model input from the previous question. These views are valuable during evaluation because a correct-looking answer should still be checked against the evidence that produced it.

For an ordinary question, the loop calls `rag_query()` inside a `try` block. The returned values replace the stored previous result, and the notebook prints retrieval details, selected evidence, generation diagnostics, the RAG answer, all retrieved sources, and the prompt sent to Qwen. This output is intentionally detailed because the activity examines both retrieval and answer generation.

If an exception occurs, the cell prints the error and explains that the question could not be processed instead of terminating the entire notebook. The user can then enter another question or inspect the setup. Exiting this loop does not cancel the Excel download started in Component I because that browser-side action was created earlier and operates separately.


In [ ]:
# ============================================================
# 10. CONTINUOUS QUESTION-ANSWER LOOP
# ============================================================

print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask English questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Store the most recent query results.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # --------------------------------------------------------
    # EXIT CONDITION
    # --------------------------------------------------------

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # --------------------------------------------------------
    # IGNORE EMPTY QUESTIONS
    # --------------------------------------------------------

    if not user_question:

        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: SOURCES
    # --------------------------------------------------------

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Reranker {result['score']:.4f} | "
                f"FAISS rank {result['faiss_rank']}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: PROMPT
    # --------------------------------------------------------

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print(last_prompt)

        continue

    # --------------------------------------------------------
    # PROCESS QUESTION
    # --------------------------------------------------------

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # ----------------------------------------------------
        # DISPLAY ANSWER
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Reranker {result['score']:.4f} "
                f"| FAISS rank {result['faiss_rank']}"
            )

            print("-" * 70)
            print(result["text"])

        # ----------------------------------------------------
        # DISPLAY PROMPT
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print(prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")






CONTINUOUS RAG QUESTION ANSWERING
Ask English questions about the indexed document.
Type 'exit' to terminate.
Type 'sources' to display sources from the last query.
Type 'prompt' to display the last generated prompt.


Question: What is the total pay rate for work done during a regular holiday for the first eight hours?

--- RETRIEVAL QUERY ---
What is the total pay rate for work done during a regular holiday for the first eight hours?

--- RERANKING DIAGNOSTICS ---
FAISS candidates: 30
Rank 1: Page 28 | Chunk 77 | Reranker 7.7176 | FAISS rank 2
Rank 2: Page 32 | Chunk 89 | Reranker 7.3599 | FAISS rank 3
Rank 3: Page 33 | Chunk 91 | Reranker 5.2962 | FAISS rank 1

--- PROMPT DIAGNOSTICS ---
Original question: What is the total pay rate for work done during a regular holiday for the first eight hours?
Retrieved chunks: 3
Chunks included: 3
Final prompt tokens: 899
Input token limit: 4096

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 28, Chunk 77]
18 provision in the Collective Bargain

### 11. Mark the end of the program

After `exit` breaks the interactive loop, this final cell prints a clear completion banner. It does not delete the index, unload the models, or change the saved files. Its purpose is simply to show that the session reached a normal ending rather than stopping because of an error.

Separating this message from the loop also makes the final control flow easy to follow. Once the loop finishes, execution moves here and the notebook visibly confirms that the RAG program has ended.


In [ ]:
# ============================================================
# 11. END
# ============================================================

print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)




RAG PROGRAM FINISHED
